# Spark architecture — see it, run it, explain it

**Goal:** explain where your code runs, where data lives, and how one query becomes parallel work.

**Study loop:** predict → run one cell → inspect its output/Spark UI → explain it aloud.
Allow 45–60 minutes; stop after any experiment and resume later. Every example uses a tiny synthetic orders dataset. No credentials, downloads, cloud paths, or paid services are needed by the code itself. Running on paid compute still uses that compute.

| Route | What to do |
|---|---|
| Databricks | Import this `.ipynb`, attach compute, and run top to bottom. Classic compute is best for the full architecture/UI lab. |
| Local Jupyter | Use the repository environment (`pip install -r requirements.txt`), install a compatible Java runtime, launch `jupyter lab`, and open this file. |
| Serverless / Spark Connect | The DataFrame experiments are the main route. SparkContext, RDDs, caching, and some settings can be restricted; optional probes report or skip those restrictions. |

Local mode demonstrates partitions and tasks on **one machine**, not a real multi-machine cluster. Do not infer cluster speed from this tiny lab.

Companion reading: [cluster architecture note](../01-cluster-architecture-deep-dive.md).

## 1. Start with the picture

A cluster is a group of machines. A **worker node is a machine**; an **executor is a process** running on it. The **driver coordinates**; executors perform the distributed processing. The cluster manager allocates resources, rather than deciding the SQL logic.


![Spark architecture: driver, cluster manager, two workers, and storage](attachment:spark_architecture.png)

This is a conceptual example with one executor per worker. Other deployments can place multiple executors on a worker. Arrows summarize communication, not every protocol message; the manager can launch executors on both workers.

**Say aloud:** “The driver sends work. Executors read and process the data. Storage keeps the durable files.”


## 2. Start or reuse Spark

`SparkSession` is the entry point for DataFrames and SQL. In classic Spark it uses a `SparkContext` for the application's cluster connection. In Spark Connect, the Python client communicates with a remote Spark service; your client is not itself the remote driver.

The next cell reuses Databricks' existing `spark`. In a fresh local kernel it creates `local[4]`: four task threads on one machine. It does **not** request four executors or four computers.

When supported, a separate SQL session isolates this lab's SQL settings. We retain the previous settings and restore them at the end, even if a separate session is unavailable.


In [0]:
from pyspark.sql import SparkSession, functions as F

created_here = 'spark' not in globals()
if created_here:
    spark = (
        SparkSession.builder
        .master('local[4]')
        .appName('Ishan-Spark-Architecture-Lab')
        .config('spark.sql.shuffle.partitions', '4')
        .getOrCreate()
    )

try:
    lab = spark.newSession()
except Exception as exc:
    lab = spark
    print('Using existing SQL session:', type(exc).__name__)

original_settings = {}
cached_orders = None
for key, value in {
    'spark.sql.shuffle.partitions': '4',
    'spark.sql.adaptive.enabled': 'false',
}.items():
    try:
        original_settings[key] = lab.conf.get(key)
        lab.conf.set(key, value)
        print(key, '=', lab.conf.get(key))
    except Exception as exc:
        print('Setting unavailable:', key, type(exc).__name__)

print('Spark version:', lab.version)
print('AQE is disabled where allowed only to make stages easier to inspect.')
print('If AQE remains enabled, Spark can change partition counts during execution.')


### Inspect the application, not just the notebook

This optional introspection uses classic `SparkContext`. A restriction here does not mean the DataFrame route is broken.

**UI check:** on Databricks, open the attached compute's **Spark UI** when available. Locally, use the URL printed below. Keep the application alive while inspecting it. On serverless, use the query profile/available execution view; it may not expose all classic tabs.


In [0]:
sc = None
try:
    sc = lab.sparkContext
    print('Application ID:', sc.applicationId)
    print('Master:', sc.master)
    print('Default parallelism:', sc.defaultParallelism)
    print('Spark UI:', sc.uiWebUrl)
except Exception as exc:
    sc = None
    print('Classic SparkContext unavailable:', type(exc).__name__)
    print('Continue with the DataFrame experiments.')


## 3. Create orders in eight input partitions

We generate rows directly so storage credentials do not distract from architecture. In production the input could instead be Parquet/Delta files in ADLS. **This lab does not demonstrate an actual ADLS read.**

| Expression | Why it exists |
|---|---|
| `range(..., numPartitions=8)` | Requests eight input partitions without collecting rows in Python. |
| `id % 3` | Assigns one of three cities. |
| `id % 5` | Marks every fifth order cancelled. |
| `id % 100 + 1` | Creates a small integer order amount. |

**Predict:** will assigning `orders` scan all its rows? No: these transformations build a plan. Source/schema discovery can do work for other sources, but this synthetic input requires no file discovery.


In [0]:
N = 24_000
orders = (
    lab.range(0, N, 1, numPartitions=8)
    .withColumnRenamed('id', 'order_id')
    .withColumn(
        'city',
        F.element_at(F.array(F.lit('Delhi'), F.lit('Meerut'), F.lit('Noida')),
                     (F.col('order_id') % 3 + 1).cast('int'))
    )
    .withColumn('status', F.when(F.col('order_id') % 5 == 0, 'Cancelled')
                .otherwise('Delivered'))
    .withColumn('amount', F.col('order_id') % 100 + 1)
)
print('Plan created. No action requested by this cell.')
orders.printSchema()


### Observe a partition

A partition is a logical portion of a distributed dataset, not necessarily one file or one machine. A task processes one partition **within a particular stage**. Partitions can be processed by different executors on different runs.

The next cell is an action. It computes partition IDs before the diagnostic aggregation. That aggregation and its final sort introduce their own exchanges: do not mistake them for shuffles in the original `range` input.

**Expect:** eight populated input partitions, 3,000 rows each. The IDs shown describe this point in this plan; they are not permanent row IDs.


In [0]:
partition_sizes = (
    orders.select(F.spark_partition_id().alias('input_partition'))
    .groupBy('input_partition').count()
)
partition_sizes.orderBy('input_partition').show()


## 4. Follow a narrow transformation

`filter` and this `select` can process rows independently in each input partition. They do not require moving rows between partitions. That makes them **narrow transformations**.

`explain('formatted')` describes the physical plan without requesting the full result. Read the plan from the source upward: `Range` → `Project` → `Filter` (the optimizer may combine/reorder operations).

**UI check:** note the current job list, run the action cell, then inspect its new SQL execution. The `count()` includes a final aggregation exchange even though the filter itself is narrow. Read operators rather than assuming the entire action has no shuffle.


In [0]:
delivered = orders.filter(F.col('status') == 'Delivered').select(
    'order_id', 'city', 'amount'
)
delivered.explain('formatted')


In [0]:
delivered_count = delivered.count()
print('Delivered orders:', delivered_count)
assert delivered_count == 19_200


## 5. Follow a shuffle: total sales per city

Delhi rows can start in several partitions. Spark typically computes partial totals near the input, exchanges those contributions by city, and combines them into final totals. This data redistribution is a **shuffle**.

The driver coordinates this work. It does not receive all 24,000 source rows to add them itself.

**Predict:** how many result rows will there be? Three — one per city.

**Inspect the plan:** find `Exchange` and partial/final `HashAggregate` (operator names can vary by engine). A shuffle separates scheduler stages. A stage contains tasks, generally one per partition processed by that stage. An action can produce more than one job; do not memorize “one action = exactly one job.”


In [0]:
sales = delivered.groupBy('city').agg(
    F.count('*').alias('delivered_orders'),
    F.sum('amount').alias('total_sales')
)
sales.explain('formatted')


In [0]:
# Three city results are safe to bring to the driver in this fixed-size lab.
sales_rows = sales.collect()
for row in sorted(sales_rows, key=lambda row: row['city']):
    print(row.asDict())

assert len(sales_rows) == 3
assert sum(row['delivered_orders'] for row in sales_rows) == 19_200
assert sum(row['total_sales'] for row in sales_rows) == 979_200


### Open Spark UI now

| View | Find | What it tells you |
|---|---|---|
| SQL / DataFrame | This sales aggregation and its `Exchange` | Where redistribution occurs in the query. |
| Jobs → Stages | Stage task counts | How partitions become runnable tasks. |
| Stage details | Shuffle write/read metrics | How much intermediate data is exchanged. |
| Executors | Executor IDs, cores, completed tasks | Which processes actually did the work. |

With AQE disabled and the requested settings accepted, the input has eight partitions and the city aggregation requests four shuffle partitions. Four partitions can contain only three cities, so some partitions may be empty. Hash collisions can also put different cities together. AQE, Photon, and runtime optimizations can change the visible plan and task counts.

**Write your observations:**

| Observation | Your value |
|---|---|
| Input stage task count | |
| Final aggregation stage task count | |
| Number of executor processes | |
| Shuffle write / read bytes | |

Do not include an extra `orderBy` when comparing the plan above: a global sort can add another shuffle.


## 6. See rows move when repartitioning

`repartition(4, 'city')` explicitly requests a hash shuffle. All rows for the same city land in the same output partition for this operation; multiple cities can share a partition. Capture IDs **after** repartitioning.

The summary below adds a separate diagnostic aggregation and sort. It lists only nonempty output partitions; missing IDs do not mean the requested partition did not exist.


In [0]:
by_city = delivered.repartition(4, 'city')
by_city.explain('formatted')
placement = by_city.select('city', F.spark_partition_id().alias('output_partition'))
placement.groupBy('output_partition', 'city').count().orderBy(
    'output_partition', 'city'
).show()


## 7. Cores, partitions, and task concurrency

Suppose a cluster gives your application **3 executors × 4 cores**, with one CPU required per task. There are 60 tasks ready in one stage.

**Predict before running:** how many can run simultaneously? What happens to the others?

This is a capacity model, not a measurement of your current cluster. Resource availability, task CPU requirements, GPU needs, and scheduling constraints can reduce concurrency. Sixty partitions do not create sixty cores.


In [0]:
import math

executors = 3
cores_per_executor = 4
cpus_per_task = 1
ready_tasks = 60
slots = executors * (cores_per_executor // cpus_per_task)
running_at_once = min(slots, ready_tasks)
ideal_waves = math.ceil(ready_tasks / slots)
print('Maximum concurrent tasks in this model:', running_at_once)
print('Ideal equal-duration waves:', ideal_waves)
print('Try cpus_per_task = 2 and explain why concurrency decreases.')


### Optional: see Python worker identities on classic Spark

Enable only on local/classic compute where RDDs are allowed. This probe returns one tiny record per partition. `host` and `python_pid` identify a Python worker process executing task code — **not the JVM executor ID**. Python workers may be reused; several processes can share a host. Use the Executors tab for actual executor IDs.

Local mode will show one host. Matching hostnames do not prove the presence of multiple machines, and a Python worker is not an executor.


In [0]:
RUN_PROCESS_PROBE = False

def describe_partition(partition_id: int, values: object) -> object:
    import os
    import socket
    from pyspark import TaskContext

    context = TaskContext.get()
    yield (partition_id, sum(1 for _ in values), socket.gethostname(),
           os.getpid(), context.stageId(), context.attemptNumber())

if RUN_PROCESS_PROBE and sc is not None:
    records = sc.parallelize(range(80), 8).mapPartitionsWithIndex(
        describe_partition
    ).collect()
    for record in records:
        print(dict(zip(
            ['partition', 'rows', 'host', 'python_pid', 'stage', 'attempt'], record
        )))
else:
    print('Optional RDD probe skipped. Enable on compatible classic/local compute.')


## 8. Where cached data lives

`cache()` marks a DataFrame for reuse. An action populates the cache. On a cluster, cached blocks are held by executors, using the storage level's memory/disk behavior. The driver tracks metadata; it does not hold the entire distributed cache.

Execution memory supports joins, sorts, and aggregations; storage memory supports cached blocks. They share a managed region. Python workers/native allocations also consume memory outside the executor JVM heap. Not every PySpark operation executes in Python: built-in SQL expressions usually execute in the Spark engine.

**Inspect:** the second action's query plan may show `InMemoryTableScan`. Check the Storage tab for materialized blocks. On a tiny dataset, caching may be slower; timings here are not a benchmark. Serverless can restrict caching, so this optional experiment can be skipped.


In [0]:
cached_orders = None
try:
    cached_orders = delivered.cache()
    print('Rows used to populate cache:', cached_orders.count())
except Exception as exc:
    print('Optional cache experiment unavailable:', type(exc).__name__)
    if cached_orders is not None:
        try:
            cached_orders.unpersist()
        except Exception:
            pass
    cached_orders = None

if cached_orders is not None:
    print('Storage level:', cached_orders.storageLevel)
    reused_query = cached_orders.filter(F.col('city') == 'Delhi')
    reused_query.explain('formatted')
    print('Delivered Delhi rows:', reused_query.count())


## 9. Driver memory: why collect can hurt

`collect()` transfers every result row into the client/driver-side Python process in classic PySpark. In Spark Connect the result comes to the client; the remote driver is a different process. Neither is a good place to receive unbounded data.

Our aggregation has exactly three rows, so collecting it is safe. Collecting a huge raw orders table can exhaust memory. `show(5)` bounds the displayed result, but upstream operations such as an aggregation may still process the full input.

**Run safely:** collect only a bounded five-row sample. Do not simulate an out-of-memory failure on shared compute.


In [0]:
sample_rows = orders.limit(5).collect()
print('Rows returned to this Python process:', len(sample_rows))
for row in sample_rows:
    print(row.asDict())


## 10. Diagnose by component

| Symptom | First evidence to inspect | Possible next step |
|---|---|---|
| Driver/client runs out of memory | Large `collect` / `toPandas`, result size | Bound the result or write distributed output. |
| One task is much slower | Task duration and input/shuffle size distribution | Confirm skew; review partitioning and hot keys. |
| Executor is lost | Executor logs, container exit reason, memory metrics | Confirm heap/native/Python pressure before resizing. |
| High shuffle traffic | SQL exchanges and shuffle read/write | Filter early; review joins, grouping, and partitioning. |
| Spill to disk | Stage spill metrics and executor disk | Check per-task working set; spill is not automatically failure. |

Lost executor tasks can often be retried/recomputed from lineage if the underlying inputs remain available. A deterministic code error is likely to fail again. Driver failure interrupts the running application; some deployment systems can restart it, which is not the same as retrying an individual task. Durable streaming checkpoints are a separate recovery mechanism.

## 11. Change one thing, explain the result

1. Change input `numPartitions` from 8 to 16. Re-run from dataset creation. What changes in the input stage? The row totals should remain the same.
2. Change the requested shuffle partitions from 4 to 8 in setup. Rebuild `sales`. Explain why eight partitions do not imply eight result cities.
3. In the concurrency model, change CPUs per task to 2. Explain the new capacity.
4. Re-enable AQE where supported. Re-run the query and compare its final plan in the UI. Counts can change; the business result should not.
5. Draw the architecture from memory and mark where `collect`, cache, and shuffle put data.

**Avoid:** using this 24,000-row dataset to choose a production cluster size. There is no universal 5-core or 32-GB executor limit; use workload metrics and runtime constraints.

## 12. Explain it aloud, then check yourself

> A Spark application has a driver and executors. The driver coordinates execution, and the cluster manager allocates resources. Executors run tasks against partitions. A shuffle redistributes intermediate data between partitions, often across executors. The Spark UI helps me inspect stages, tasks, shuffle, and executor behavior.

| Question | Answer — cover this column first |
|---|---|
| Is an executor a machine? | No. It is a process hosted on a worker machine. |
| Does a partition equal a core? | No. Partitions are units of data; cores provide execution capacity. |
| Where does a task execute? | In an executor; Python task logic may use an associated Python worker. |
| Who schedules tasks? | The driver, using its schedulers. |
| What do Catalyst and DAG scheduling do? | Catalyst optimizes structured query plans; the DAG scheduler organizes executable work into stages. |
| What does the task scheduler do? | Assigns runnable tasks to available executor resources and handles task attempts. |
| Where is cache metadata tracked? | Driver-side block management tracks block locations; executors manage their blocks. Shuffle tracking has additional components. |
| Must all input fit in RAM? | No. Spark can process partitions and spill supported intermediate work to disk. |
| Does local[4] mean four workers? | No. Four local task threads on one machine. |

**Your five-sentence explanation:** replace this paragraph with your own account of driver, manager, executor, partition, and shuffle.

## 13. Clean up

Run this cell even if you stop early: it releases this lab's cache and restores the SQL settings it saved. It stops Spark only when this notebook created the local session. It leaves an existing Databricks session running. If a prior cell failed, return here before starting the lab again.


In [0]:
if globals().get('cached_orders') is not None:
    cached_orders.unpersist()
    print('Lab cache released.')

for key, value in original_settings.items():
    try:
        lab.conf.set(key, value)
    except Exception as exc:
        print('Could not restore setting:', key, type(exc).__name__)

if created_here:
    spark.stop()
    del spark
    print('Notebook-created local Spark session stopped.')
else:
    print('Existing Spark session left running.')


## Official references

These link to Apache Spark 3.5.7 documentation. Local verification targets that release; Databricks and other Spark versions may expose different UI labels, plans, and restrictions.

- [Cluster overview: driver, executors, manager, jobs/stages/tasks](https://spark.apache.org/docs/3.5.7/cluster-overview.html)
- [RDD guide: lazy execution, partitions, shuffle, persistence](https://spark.apache.org/docs/3.5.7/rdd-programming-guide.html)
- [SQL performance tuning: caching, shuffle partitions, AQE](https://spark.apache.org/docs/3.5.7/sql-performance-tuning.html)
- [Web UI: jobs, stages, storage, environment, executors, SQL](https://spark.apache.org/docs/3.5.7/web-ui.html)
- [Memory management and tuning](https://spark.apache.org/docs/3.5.7/tuning.html)
- [Configuration: task CPUs, worker reuse, memory, deployment](https://spark.apache.org/docs/3.5.7/configuration.html)
- [Spark Connect: separation of client and server](https://spark.apache.org/docs/3.5.7/spark-connect-overview.html)

Book companions: [LS Ch.2, Ch.3]; [HPS Ch.2, Ch.3]. See the repository's [book map](../../BOOK_MAP.md).
